# GAN-based Image Denoising & Multi-Angle View Generation
**BUBT Capstone Project** — Md. Zehad Khan et al.

## Datasets required (attach before running):
- `jessicali9530/celeba-dataset` → Phase 1
- `maulidio16/300w-lp` → Phase 2

## Steps:
1. GPU on: Settings → Accelerator → GPU T4 x2
2. Attach both datasets
3. Run all cells

In [ ]:
# Cell 1: GPU Check
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = torch.cuda.is_available()

In [ ]:
# Cell 2: Install dependencies
!pip install -q scipy

In [ ]:
# Cell 3: Imports
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.utils import save_image
from PIL import Image
from scipy.io import loadmat
import matplotlib.pyplot as plt

# torch.cuda.amp.autocast is deprecated in PyTorch 2.x — use new API
autocast = lambda enabled=True: torch.amp.autocast('cuda', enabled=enabled)

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints/phase2', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
os.makedirs('/kaggle/working/results/phase2', exist_ok=True)
print('Directories ready.')

---
## Phase 1 — Image Denoising (U-Net + PatchGAN)

In [ ]:
# Cell 4: Phase 1 Config
P1_DATA_DIR  = '/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba'
P1_CKPT_DIR  = '/kaggle/working/checkpoints'
P1_RES_DIR   = '/kaggle/working/results'

IMAGE_SIZE    = 128
NOISE_STD     = 0.10
BATCH_SIZE    = 32        # increased from 16
NUM_EPOCHS_P1 = 100       # increased from 50
WARMUP_EPOCHS = 5
LR            = 1e-4
BETA1, BETA2  = 0.5, 0.999

LAMBDA_L1    = 100.0
LAMBDA_PERC  = 10.0
LAMBDA_SSIM  = 10.0
LAMBDA_ADV   = 1.0

TRAIN_SIZE   = 2000
VAL_SIZE     = 400
print('Phase 1 config ready.')

In [ ]:
# Cell 5: CelebA Dataset (with augmentation)
class CelebADataset(Dataset):
    def __init__(self, root_dir, split='train', image_size=128, noise_std=0.10,
                 train_size=2000, val_size=400):
        self.noise_std = noise_std
        all_imgs = sorted([
            os.path.join(root_dir, f) for f in os.listdir(root_dir)
            if f.lower().endswith(('.jpg', '.png'))
        ])[:train_size + val_size]
        self.paths = all_imgs[:train_size] if split == 'train' else all_imgs[train_size:]

        # Augmentation only for training
        if split == 'train':
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                transforms.ToTensor(),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
            ])

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        clean = self.tf(Image.open(self.paths[idx]).convert('RGB'))
        noisy = torch.clamp(clean + torch.randn_like(clean) * self.noise_std, 0, 1)
        return noisy, clean

train_ds = CelebADataset(P1_DATA_DIR, 'train', IMAGE_SIZE, NOISE_STD, TRAIN_SIZE, VAL_SIZE)
val_ds   = CelebADataset(P1_DATA_DIR, 'val',   IMAGE_SIZE, NOISE_STD, TRAIN_SIZE, VAL_SIZE)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Cell 6: U-Net Generator
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=not use_bn)]
        if use_bn: layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
                  nn.BatchNorm2d(out_ch)]
        if dropout: layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip): return torch.cat([self.block(x), skip], dim=1)

class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DownBlock(3, 64, use_bn=False)
        self.enc2 = DownBlock(64, 128)
        self.enc3 = DownBlock(128, 256)
        self.enc4 = DownBlock(256, 512)
        self.enc5 = DownBlock(512, 512)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, 4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.dec1 = UpBlock(512, 512, dropout=True)
        self.dec2 = UpBlock(1024, 512, dropout=True)
        self.dec3 = UpBlock(1024, 256, dropout=True)
        self.dec4 = UpBlock(512, 128)
        self.dec5 = UpBlock(256, 64)
        self.out  = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, stride=2, padding=1), nn.Sigmoid())

    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(e1); e3=self.enc3(e2)
        e4=self.enc4(e3); e5=self.enc5(e4); b=self.bottleneck(e5)
        d=self.dec1(b,e5); d=self.dec2(d,e4); d=self.dec3(d,e3)
        d=self.dec4(d,e2); d=self.dec5(d,e1)
        return self.out(d)

print('U-Net Generator defined.')

In [ ]:
# Cell 7: PatchGAN Discriminator (with Spectral Normalization)
from torch.nn.utils import spectral_norm

class PatchGAN(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(ic, oc, s, bn):
            l = [spectral_norm(nn.Conv2d(ic, oc, 4, stride=s, padding=1, bias=not bn))]
            if bn: l.append(nn.BatchNorm2d(oc))
            l.append(nn.LeakyReLU(0.2, inplace=True))
            return l
        self.model = nn.Sequential(
            *blk(6,64,2,False), *blk(64,128,2,True),
            *blk(128,256,2,True), *blk(256,512,1,True),
            spectral_norm(nn.Conv2d(512,1,4,stride=1,padding=1)))
    def forward(self, noisy, tgt):
        return self.model(torch.cat([noisy, tgt], dim=1))

print('PatchGAN Discriminator (Spectral Norm) defined.')

In [ ]:
# Cell 8: Loss Functions (Phase 1)
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.s1 = nn.Sequential(*list(vgg)[:4])
        self.s2 = nn.Sequential(*list(vgg)[4:9])
        self.s3 = nn.Sequential(*list(vgg)[9:16])
        for p in self.parameters(): p.requires_grad = False
        self.c = nn.L1Loss()
    def forward(self, g, t):
        g1=self.s1(g); t1=self.s1(t)
        g2=self.s2(g1); t2=self.s2(t1)
        g3=self.s3(g2); t3=self.s3(t2)
        return (self.c(g1,t1)+self.c(g2,t2)+self.c(g3,t3))/3

class SSIMLoss(nn.Module):
    def __init__(self, ws=11, sig=1.5):
        super().__init__()
        coords = torch.arange(ws, dtype=torch.float32) - ws//2
        g = torch.exp(-(coords**2)/(2*sig**2)); g = g/g.sum()
        self.register_buffer('w', (g[:,None]*g[None,:]).unsqueeze(0).unsqueeze(0))
        self.ws = ws
    def _ssim(self, x, y):
        C1,C2 = 0.01**2, 0.03**2
        p = self.ws//2
        win = self.w.expand(x.size(1),1,-1,-1)
        mx=F.conv2d(x,win,padding=p,groups=x.size(1))
        my=F.conv2d(y,win,padding=p,groups=x.size(1))
        sx=F.conv2d(x*x,win,padding=p,groups=x.size(1))-mx**2
        sy=F.conv2d(y*y,win,padding=p,groups=x.size(1))-my**2
        sxy=F.conv2d(x*y,win,padding=p,groups=x.size(1))-mx*my
        return ((2*mx*my+C1)*(2*sxy+C2)/((mx**2+my**2+C1)*(sx+sy+C2))).mean()
    def forward(self, g, t): return 1.0 - self._ssim(g, t)

class GANLoss(nn.Module):
    def __init__(self): super().__init__(); self.c = nn.BCEWithLogitsLoss()
    def forward(self, p, real): return self.c(p, torch.ones_like(p) if real else torch.zeros_like(p))

perc_loss = PerceptualLoss().to(DEVICE)
ssim_loss = SSIMLoss().to(DEVICE)
gan_loss  = GANLoss().to(DEVICE)
l1_loss   = nn.L1Loss()
print('Phase 1 losses ready.')

In [ ]:
# Cell 9: Metrics
def compute_psnr(gen, tgt):
    mse = F.mse_loss(gen.detach(), tgt.detach()).item()
    return float('inf') if mse == 0 else 20*math.log10(1.0) - 10*math.log10(mse)

def compute_ssim(x, y, ws=11, sig=1.5):
    x,y = x.detach(), y.detach()
    C1,C2 = 0.01**2, 0.03**2
    coords = torch.arange(ws,dtype=x.dtype,device=x.device)-ws//2
    g = torch.exp(-(coords**2)/(2*sig**2)); g=g/g.sum()
    win=(g[:,None]*g[None,:]).unsqueeze(0).unsqueeze(0).expand(x.size(1),1,-1,-1)
    p=ws//2
    mx=F.conv2d(x,win,padding=p,groups=x.size(1))
    my=F.conv2d(y,win,padding=p,groups=x.size(1))
    sx=F.conv2d(x*x,win,padding=p,groups=x.size(1))-mx**2
    sy=F.conv2d(y*y,win,padding=p,groups=x.size(1))-my**2
    sxy=F.conv2d(x*y,win,padding=p,groups=x.size(1))-mx*my
    return ((2*mx*my+C1)*(2*sxy+C2)/((mx**2+my**2+C1)*(sx+sy+C2))).mean().item()

print('Metrics ready.')

In [ ]:
# Cell 10: Phase 1 Training (with LR Scheduler)
G1 = UNetGenerator().to(DEVICE)
D1 = PatchGAN().to(DEVICE)
opt_g1 = optim.Adam(G1.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_d1 = optim.Adam(D1.parameters(), lr=LR, betas=(BETA1, BETA2))

# CosineAnnealingLR: smoothly decays LR from 1e-4 → 1e-6
scheduler_g1 = optim.lr_scheduler.CosineAnnealingLR(opt_g1, T_max=NUM_EPOCHS_P1, eta_min=1e-6)
scheduler_d1 = optim.lr_scheduler.CosineAnnealingLR(opt_d1, T_max=NUM_EPOCHS_P1, eta_min=1e-6)

scaler_g1 = GradScaler(enabled=AMP)
scaler_d1 = GradScaler(enabled=AMP)

best_psnr = 0.0
p1_history = {'g_loss': [], 'd_loss': [], 'psnr': [], 'ssim': []}

for epoch in range(1, NUM_EPOCHS_P1 + 1):
    G1.train(); D1.train()
    warmup = epoch <= WARMUP_EPOCHS
    g_tot = d_tot = 0.0
    t0 = time.time()

    for noisy, clean in train_loader:
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)

        # D step (skipped during warmup)
        if not warmup:
            opt_d1.zero_grad()
            with autocast(enabled=AMP):
                fake = G1(noisy).detach()
                ld = 0.5*(gan_loss(D1(noisy,clean),True)+gan_loss(D1(noisy,fake),False))
            scaler_d1.scale(ld).backward()
            scaler_d1.step(opt_d1); scaler_d1.update()
            d_tot += ld.item()

        # G step
        opt_g1.zero_grad()
        with autocast(enabled=AMP):
            fake = G1(noisy)
            lg = LAMBDA_L1 * l1_loss(fake, clean)
            if not warmup:
                lg += LAMBDA_PERC * perc_loss(fake, clean)
                lg += LAMBDA_SSIM * ssim_loss(fake, clean)
                lg += LAMBDA_ADV  * gan_loss(D1(noisy, fake), True)
        scaler_g1.scale(lg).backward()
        scaler_g1.step(opt_g1); scaler_g1.update()
        g_tot += lg.item()

    # Step LR schedulers
    scheduler_g1.step()
    scheduler_d1.step()

    n = len(train_loader)
    g_avg = g_tot/n; d_avg = d_tot/n

    # Validation
    G1.eval(); vp=vs=0.0
    with torch.no_grad():
        for i,(noisy,clean) in enumerate(val_loader):
            noisy,clean = noisy.to(DEVICE),clean.to(DEVICE)
            with autocast(enabled=AMP): gen=G1(noisy)
            vp += compute_psnr(gen,clean); vs += compute_ssim(gen,clean)
            if i==0 and epoch%10==0:
                save_image(torch.cat([noisy[:4],gen[:4],clean[:4]],0),
                           f'{P1_RES_DIR}/epoch_{epoch:03d}.png', nrow=4)
    vp/=len(val_loader); vs/=len(val_loader)

    cur_lr = scheduler_g1.get_last_lr()[0]
    mode = 'WARMUP' if warmup else 'FULL '
    print(f'[{mode}] Ep {epoch:03d}/{NUM_EPOCHS_P1} | G={g_avg:.3f} D={d_avg:.3f} | '
          f'PSNR={vp:.2f} SSIM={vs:.4f} | LR={cur_lr:.2e} | {time.time()-t0:.1f}s')

    p1_history['g_loss'].append(g_avg); p1_history['d_loss'].append(d_avg)
    p1_history['psnr'].append(vp);      p1_history['ssim'].append(vs)

    if vp > best_psnr:
        best_psnr = vp
        torch.save({'epoch':epoch,'generator':G1.state_dict(),
                    'discriminator':D1.state_dict(),'best_psnr':best_psnr},
                   f'{P1_CKPT_DIR}/best_model.pth')

print(f'\nPhase 1 Done. Best PSNR: {best_psnr:.2f} dB')

In [ ]:
# Cell 11: Phase 1 Results Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, NUM_EPOCHS_P1+1)
axes[0].plot(epochs, p1_history['g_loss'], label='G Loss')
axes[0].plot(epochs, p1_history['d_loss'], label='D Loss')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[1].plot(epochs, p1_history['psnr'])
axes[1].set_title('Validation PSNR (dB)')
axes[2].plot(epochs, p1_history['ssim'])
axes[2].set_title('Validation SSIM')
plt.tight_layout(); plt.savefig(f'{P1_RES_DIR}/training_curves.png'); plt.show()

G1.eval()
noisy_sample, clean_sample = next(iter(val_loader))
noisy_sample = noisy_sample[:6].to(DEVICE)
with torch.no_grad():
    with torch.amp.autocast('cuda', enabled=AMP):
        gen_sample = G1(noisy_sample)

# float32 convert — AMP float16 দেয়, matplotlib সেটা support করে না
noisy_np = noisy_sample.cpu().float()
gen_np   = gen_sample.cpu().float()
clean_np = clean_sample.float()

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    axes[0,i].imshow(noisy_np[i].permute(1,2,0).clamp(0,1).numpy())
    axes[0,i].set_title('Noisy'); axes[0,i].axis('off')
    axes[1,i].imshow(gen_np[i].permute(1,2,0).clamp(0,1).numpy())
    axes[1,i].set_title('Denoised'); axes[1,i].axis('off')
    axes[2,i].imshow(clean_np[i].permute(1,2,0).clamp(0,1).numpy())
    axes[2,i].set_title('Ground Truth'); axes[2,i].axis('off')
plt.suptitle('Phase 1: Noisy → Denoised → Ground Truth')
plt.tight_layout(); plt.savefig(f'{P1_RES_DIR}/sample_results.png'); plt.show()
print(f'Final PSNR: {p1_history["psnr"][-1]:.2f} dB | SSIM: {p1_history["ssim"][-1]:.4f}')

---
## Phase 2 — Multi-Angle View Generation (Pose-conditioned cGAN)

In [ ]:
# Cell 12: Phase 2 Config
P2_DATA_DIR  = '/kaggle/input/datasets/maulidio16/300w-lp/300W_LP'
P2_CKPT_DIR  = '/kaggle/working/checkpoints/phase2'
P2_RES_DIR   = '/kaggle/working/results/phase2'

IMAGE_SIZE = 256

NUM_POSE_CLASSES = 5
POSE_ANGLES      = [-90, -45, 0, 45, 90]
BATCH_SIZE_P2    = 16
NUM_EPOCHS_P2    = 100
LR_P2            = 2e-4

LAMBDA_RECON    = 10.0
LAMBDA_IDENTITY = 5.0
LAMBDA_ADV_P2   = 1.0
LAMBDA_POSE     = 2.0
LAMBDA_CYCLE    = 10.0

print(f'300W-LP path: {P2_DATA_DIR}')
print('Phase 2 config ready.')

In [ ]:
# Cell 13: 300W-LP Pose Dataset

# Safety check — make sure Cell 12 was run
assert 'P2_DATA_DIR' in dir() or 'P2_DATA_DIR' in globals(), \
    "ERROR: Run Cell 12 (Phase 2 Config) first!"

print("P2_DATA_DIR   =", P2_DATA_DIR)
print("IMAGE_SIZE    =", IMAGE_SIZE)
print("BATCH_SIZE_P2 =", BATCH_SIZE_P2)


def yaw_to_class(yaw_deg):
    return int(np.argmin([abs(yaw_deg - a) for a in POSE_ANGLES]))


def to_onehot(cls, n=NUM_POSE_CLASSES):
    v = torch.zeros(n, dtype=torch.float32)
    v[cls] = 1.0
    return v


class PoseDataset(Dataset):

    def __init__(self, root_dir, split='train'):
        self.samples = []
        for subset in sorted(os.listdir(root_dir)):
            subset_dir = os.path.join(root_dir, subset)
            if not os.path.isdir(subset_dir):
                continue
            for fname in os.listdir(subset_dir):
                if not fname.lower().endswith('.jpg'):
                    continue
                img_path = os.path.join(subset_dir, fname)
                mat_path = os.path.join(subset_dir, fname.replace('.jpg', '.mat'))
                if not os.path.exists(mat_path):
                    continue
                try:
                    mat = loadmat(mat_path)
                    if 'Pose_Para' not in mat:
                        continue
                    yaw = float(mat['Pose_Para'][0][1]) * (180.0 / np.pi)
                    self.samples.append((img_path, yaw_to_class(yaw)))
                except Exception:
                    continue

        random.shuffle(self.samples)

        n = len(self.samples)
        n_train = int(n * 0.90)
        self.samples = self.samples[:n_train] if split == 'train' else self.samples[n_train:]

        self.tf = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, src_cls = self.samples[idx]
        img = self.tf(Image.open(img_path).convert('RGB'))
        tgt_cls = random.randint(0, NUM_POSE_CLASSES - 1)
        return img, to_onehot(src_cls), to_onehot(tgt_cls), src_cls, tgt_cls


p2_train_ds = PoseDataset(P2_DATA_DIR, split='train')
p2_val_ds   = PoseDataset(P2_DATA_DIR, split='val')

p2_train_loader = DataLoader(
    p2_train_ds, batch_size=BATCH_SIZE_P2,
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)
p2_val_loader = DataLoader(
    p2_val_ds, batch_size=BATCH_SIZE_P2,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f'Pose dataset — Train: {len(p2_train_ds)} | Val: {len(p2_val_ds)}')

In [ ]:
# Cell 14: Multi-Angle Generator (Pose-conditioned U-Net)
class PoseUpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
                  nn.BatchNorm2d(out_ch)]
        if dropout: layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip): return torch.cat([self.block(x), skip], dim=1)

class MultiAngleGenerator(nn.Module):
    def __init__(self, num_poses=5):
        super().__init__()
        self.enc1 = DownBlock(3, 64, use_bn=False)
        self.enc2 = DownBlock(64, 128)
        self.enc3 = DownBlock(128, 256)
        self.enc4 = DownBlock(256, 512)
        self.enc5 = DownBlock(512, 512)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, 4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True))
        self.pose_proj = nn.Sequential(nn.Linear(num_poses, 512), nn.ReLU(inplace=True))
        # bottleneck(512) + pose(512) = 1024 input to dec1
        self.dec1 = PoseUpBlock(1024, 512, dropout=True)
        self.dec2 = PoseUpBlock(1024, 512, dropout=True)
        self.dec3 = PoseUpBlock(1024, 256, dropout=True)
        self.dec4 = PoseUpBlock(512, 128)
        self.dec5 = PoseUpBlock(256, 64)
        self.out  = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, stride=2, padding=1), nn.Sigmoid())

    def forward(self, x, pose):
        e1=self.enc1(x); e2=self.enc2(e1); e3=self.enc3(e2)
        e4=self.enc4(e3); e5=self.enc5(e4); b=self.bottleneck(e5)
        p = self.pose_proj(pose).unsqueeze(-1).unsqueeze(-1).expand_as(b)
        b = torch.cat([b, p], dim=1)
        d=self.dec1(b,e5); d=self.dec2(d,e4); d=self.dec3(d,e3)
        d=self.dec4(d,e2); d=self.dec5(d,e1)
        return self.out(d)

print('Multi-Angle Generator defined.')

In [ ]:
# Cell 15: Pose Discriminator
class PoseDiscriminator(nn.Module):
    def __init__(self, num_poses=5):
        super().__init__()
        def blk(ic,oc,s,bn):
            l=[nn.Conv2d(ic,oc,4,stride=s,padding=1,bias=not bn)]
            if bn: l.append(nn.BatchNorm2d(oc))
            l.append(nn.LeakyReLU(0.2,inplace=True)); return l
        self.shared = nn.Sequential(
            *blk(6,64,2,False),*blk(64,128,2,True),
            *blk(128,256,2,True),*blk(256,512,1,True))
        self.patch_head = nn.Conv2d(512,1,4,stride=1,padding=1)
        self.pose_head  = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512,128), nn.LeakyReLU(0.2,inplace=True),
            nn.Linear(128, num_poses))
    def forward(self, src, tgt):
        x = self.shared(torch.cat([src,tgt],dim=1))
        return self.patch_head(x), self.pose_head(x)

print('Pose Discriminator defined.')

In [ ]:
# Cell 16: Phase 2 Loss Functions
class IdentityLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.feat = nn.Sequential(*list(vgg)[:16])
        for p in self.parameters(): p.requires_grad = False
        self.c = nn.L1Loss()
    def forward(self, g, s): return self.c(self.feat(g), self.feat(s))

identity_loss = IdentityLoss().to(DEVICE)
pose_ce_loss  = nn.CrossEntropyLoss()
bce_loss      = nn.BCEWithLogitsLoss()
print('Phase 2 losses ready.')

In [ ]:
# Cell 17: Phase 2 Training
G2 = MultiAngleGenerator(num_poses=NUM_POSE_CLASSES).to(DEVICE)
D2 = PoseDiscriminator(num_poses=NUM_POSE_CLASSES).to(DEVICE)

# Transfer Phase 1 encoder weights to Phase 2 generator
p1_ckpt = torch.load(f'{P1_CKPT_DIR}/best_model.pth', map_location=DEVICE)
p1_state = p1_ckpt['generator']
g2_state = G2.state_dict()
matched = {k:v for k,v in p1_state.items() if k in g2_state and g2_state[k].shape==v.shape}
g2_state.update(matched); G2.load_state_dict(g2_state)
print(f'Transferred {len(matched)} layers from Phase 1 encoder.')

opt_g2 = optim.Adam(G2.parameters(), lr=LR_P2, betas=(BETA1, BETA2))
opt_d2 = optim.Adam(D2.parameters(), lr=LR_P2, betas=(BETA1, BETA2))
scaler_g2 = GradScaler(enabled=AMP)
scaler_d2 = GradScaler(enabled=AMP)

best_g2_loss = float('inf')
p2_history = {'g_loss': [], 'd_loss': []}

for epoch in range(1, NUM_EPOCHS_P2 + 1):
    G2.train(); D2.train()
    g_tot = d_tot = 0.0
    t0 = time.time()

    for src, src_pose, tgt_pose, src_cls, tgt_cls in p2_train_loader:
        src      = src.to(DEVICE)
        tgt_pose = tgt_pose.to(DEVICE)
        src_cls  = src_cls.to(DEVICE)
        tgt_cls  = tgt_cls.to(DEVICE)
        same_mask = (src_cls == tgt_cls)

        # D step
        opt_d2.zero_grad()
        with autocast(enabled=AMP):
            fake = G2(src, tgt_pose).detach()
            rp, rpl = D2(src, src)
            fp, fpl = D2(src, fake)
            ld = (0.5*(bce_loss(rp,torch.ones_like(rp))+bce_loss(fp,torch.zeros_like(fp)))
                  + LAMBDA_POSE * pose_ce_loss(rpl, src_cls))
        scaler_d2.scale(ld).backward()
        scaler_d2.step(opt_d2); scaler_d2.update()
        d_tot += ld.item()

        # G step
        opt_g2.zero_grad()
        with autocast(enabled=AMP):
            fake = G2(src, tgt_pose)
            fp, fpl = D2(src, fake)
            lg  = LAMBDA_ADV_P2   * bce_loss(fp, torch.ones_like(fp))
            lg += LAMBDA_IDENTITY * identity_loss(fake, src)
            lg += LAMBDA_POSE     * pose_ce_loss(fpl, tgt_cls)
            if same_mask.any():
                lg += LAMBDA_RECON * l1_loss(fake[same_mask], src[same_mask])
        scaler_g2.scale(lg).backward()
        scaler_g2.step(opt_g2); scaler_g2.update()
        g_tot += lg.item()

    n = len(p2_train_loader)
    g_avg = g_tot/n; d_avg = d_tot/n
    p2_history['g_loss'].append(g_avg); p2_history['d_loss'].append(d_avg)
    print(f'Ep {epoch:03d}/{NUM_EPOCHS_P2} | G={g_avg:.3f} D={d_avg:.3f} | {time.time()-t0:.1f}s')

    if g_avg < best_g2_loss:
        best_g2_loss = g_avg
        torch.save({'epoch':epoch,'generator':G2.state_dict(),
                    'discriminator':D2.state_dict()},
                   f'{P2_CKPT_DIR}/best_model_phase2.pth')

    # Save multi-angle samples
    if epoch % 10 == 0:
        G2.eval()
        sample_src = next(iter(p2_val_loader))[0][:4].to(DEVICE)
        all_views = [sample_src]
        with torch.no_grad():
            for ai in range(NUM_POSE_CLASSES):
                p = torch.zeros(4, NUM_POSE_CLASSES).to(DEVICE); p[:,ai]=1.0
                with autocast(enabled=AMP): gen=G2(sample_src, p)
                all_views.append(gen)
        os.makedirs(f'{P2_RES_DIR}/epoch_{epoch:03d}', exist_ok=True)
        save_image(torch.cat(all_views,0),
                   f'{P2_RES_DIR}/epoch_{epoch:03d}/multiangle.png', nrow=4)
        G2.train()

print(f'\nPhase 2 Done. Best G loss: {best_g2_loss:.4f}')

In [ ]:
# Cell 18: Phase 2 Multi-Angle Visualization
G2.eval()
sample_src = next(iter(p2_val_loader))[0][:3].to(DEVICE)
ANGLE_LABELS = ['-90°', '-45°', '0° (front)', '+45°', '+90°']

fig, axes = plt.subplots(3, 6, figsize=(20, 10))
with torch.no_grad():
    for row in range(3):
        # Input image — float32 for matplotlib
        axes[row, 0].imshow(sample_src[row].cpu().float().permute(1,2,0).clamp(0,1).numpy())
        axes[row, 0].set_title('Input'); axes[row, 0].axis('off')

        for ai in range(NUM_POSE_CLASSES):
            p = torch.zeros(1, NUM_POSE_CLASSES).to(DEVICE); p[0, ai] = 1.0
            with torch.amp.autocast('cuda', enabled=AMP):
                gen = G2(sample_src[row:row+1], p)
            # float32 convert — AMP float16 দেয়, matplotlib সেটা support করে না
            axes[row, ai+1].imshow(gen[0].cpu().float().permute(1,2,0).clamp(0,1).numpy())
            axes[row, ai+1].set_title(ANGLE_LABELS[ai])
            axes[row, ai+1].axis('off')

plt.suptitle('Phase 2: Multi-Angle View Generation\n[Input | -90° | -45° | 0° | +45° | +90°]')
plt.tight_layout()
plt.savefig(f'{P2_RES_DIR}/multiangle_final.png', dpi=150)
plt.show()
print('Multi-angle results saved!')

In [ ]:
# Cell 19: Training Curves (Phase 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep2 = range(1, NUM_EPOCHS_P2+1)
axes[0].plot(ep2, p2_history['g_loss'], label='G Loss')
axes[0].plot(ep2, p2_history['d_loss'], label='D Loss')
axes[0].set_title('Phase 2 Training Loss'); axes[0].legend()
axes[1].set_visible(False)
plt.tight_layout(); plt.savefig(f'{P2_RES_DIR}/p2_curves.png'); plt.show()

In [ ]:
# Cell 20: Download Trained Models
from IPython.display import FileLink
import shutil

# Zip both checkpoints
shutil.make_archive('/kaggle/working/models', 'zip', '/kaggle/working/checkpoints')
print('Models zipped!')
print('Download from Kaggle sidebar: Output → models.zip')
print()
print('Files saved:')
print('  /kaggle/working/checkpoints/best_model.pth         ← Phase 1 (denoising)')
print('  /kaggle/working/checkpoints/phase2/best_model_phase2.pth ← Phase 2 (multi-angle)')